# Pipeline GDELT — Bénin Insights Challenge
## Groupe 16 · iSHEERO × DataCamp Donates · 2025

**Objectif :** Transformer les données brutes GDELT extraites de BigQuery
en deux fichiers propres et enrichis prêts pour l'analyse, le modèle ML
et le dashboard interactif.

**Fichiers produits :**
- `data/processed/benin_2025_clean.csv` — Dataset complet nettoyé et enrichi
- `data/processed/benin_2025_agregat_mensuel.csv` — Agrégat mensuel pré-calculé

**Auteur :** Groupe 16 | **Date :** 2025 | **Source :** GDELT BigQuery (`ActionGeo_CountryCode = 'BN'`, YEAR = 2025)

## 1. Chargement des bibliothèques et des données brutes

On importe les bibliothèques nécessaires et on charge le fichier CSV brut
extrait de BigQuery. Ce fichier contient toutes les colonnes enrichies
définies dans le pipeline officiel du projet.

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# Chargement des données brutes
df_raw = pd.read_csv('../data/raw/gdelt_benin_2025_raw.csv')

print(f"Données brutes chargées : {df_raw.shape[0]} lignes, {df_raw.shape[1]} colonnes")
print(f"\nColonnes disponibles :")
print(df_raw.columns.tolist())
print("=== Aperçu ===")
display(df_raw.head())

print("\n=== Types de données ===")
print(df_raw.dtypes)


Données brutes chargées : 23859 lignes, 19 colonnes

Colonnes disponibles :
['SQLDATE', 'EventCode', 'EventBaseCode', 'EventRootCode', 'QuadClass', 'Actor1Name', 'Actor1CountryCode', 'Actor1Type1Code', 'Actor2Name', 'Actor2CountryCode', 'Actor2Type1Code', 'ActionGeo_FullName', 'ActionGeo_Lat', 'ActionGeo_Long', 'GoldsteinScale', 'NumMentions', 'NumArticles', 'AvgTone', 'SOURCEURL']
=== Aperçu ===


,SQLDATE,EventCode,EventBaseCode,EventRootCode,QuadClass,Actor1Name,Actor1CountryCode,Actor1Type1Code,Actor2Name,Actor2CountryCode,Actor2Type1Code,ActionGeo_FullName,ActionGeo_Lat,ActionGeo_Long,GoldsteinScale,NumMentions,NumArticles,AvgTone,SOURCEURL
0,20251231,1712,171,17,4,OPERATIVE,NaN,SPY,NaN,NaN,NaN,Benin,9.5,2.25,-9.2,10,10,-9.577465,https://dailypost.ng/2025/12/31/nscdc-arrests-...
1,20250209,16,16,1,1,CONGRESS,NaN,LEG,DEMOCRATIC PARTY,NaN,NaN,Benin,9.5,2.25,-2.0,10,10,-5.177994,https://thesun.ng/apc-dismisses-pdps-claims-on...
2,20250209,1712,171,17,4,BENIN,BEN,NaN,BENIN CITY,NGA,NaN,Benin,9.5,2.25,-9.2,4,4,-5.309735,https://dailypost.ng/2025/02/09/police-rescue-...
3,20250304,22,22,2,1,PRISONER,NaN,OPP,NaN,NaN,NaN,Benin,9.5,2.25,3.2,10,10,-2.621723,https://lanouvelletribune.info/2026/03/benin-d...
4,20250101,16,16,1,1,GOVERNOR,NaN,GOV,NaN,NaN,NaN,Benin,9.5,2.25,-2.0,3,3,-12.206573,https://punchng.com/pdp-seeks-igs-intervention...



=== Types de données ===
SQLDATE                 int64
EventCode               int64
EventBaseCode           int64
EventRootCode           int64
QuadClass               int64
Actor1Name             object
Actor1CountryCode      object
Actor1Type1Code        object
Actor2Name             object
Actor2CountryCode      object
Actor2Type1Code        object
ActionGeo_FullName     object
ActionGeo_Lat         float64
ActionGeo_Long        float64
GoldsteinScale        float64
NumMentions             int64
NumArticles             int64
AvgTone               float64
SOURCEURL              object
dtype: object


### Constat
Les données brutes contiennent **23 859 lignes** et **19 colonnes** —
toutes les colonnes définies dans le pipeline officiel sont présentes.
Le dataset couvre les événements GDELT impliquant le Bénin sur l'année 2025.
On passe maintenant au nettoyage.

## 2. Nettoyage

### 2.1 Filtrage temporel strict

On convertit SQLDATE en objet date Python puis on filtre strictement
sur l'année 2025. Cette double sécurité est nécessaire car BigQuery
filtre sur YEAR (année de détection) qui peut différer de l'année
réelle de l'événement.

In [2]:
# Conversion de la date
df_raw['SQLDATE'] = pd.to_datetime(df_raw['SQLDATE'].astype(str), format='%Y%m%d')

# Filtrage strict sur 2025
df = df_raw[df_raw['SQLDATE'].dt.year == 2025].copy()

print(f"Avant filtre  : {len(df_raw)} lignes")
print(f"Après filtre  : {len(df)} lignes")
print(f"Lignes exclues: {len(df_raw) - len(df)}")
print(f"Période       : du {df['SQLDATE'].min().date()} au {df['SQLDATE'].max().date()}")

Avant filtre  : 23859 lignes
Après filtre  : 23859 lignes
Lignes exclues: 0
Période       : du 2025-01-01 au 2025-12-31


### Constat
Le filtre temporel n'a exclu aucune ligne — BigQuery a correctement
retourné uniquement des événements de 2025. La période couverte est
du **2025-01-01 au 2025-12-31**. On travaille sur 23 859 événements.

### 2.2 Suppression des doublons stricts

On identifie et supprime les lignes identiques sur toutes les colonnes.
Ces doublons résultent d'un même article traité plusieurs fois
par le pipeline GDELT dans la même fenêtre de 15 minutes.

In [3]:
avant = len(df)
df = df.drop_duplicates()
apres = len(df)

print(f"Avant suppression : {avant} lignes")
print(f"Après suppression : {apres} lignes")
print(f"Doublons supprimés : {avant - apres} ({(avant - apres)/avant*100:.2f}%)")

Avant suppression : 23859 lignes
Après suppression : 23461 lignes
Doublons supprimés : 398 (1.67%)


### Constat
**398 doublons supprimés** (1.67%). Le dataset passe à **23 461 lignes**
propres après cette étape.

### 2.3 Gestion des valeurs manquantes

On analyse les valeurs manquantes puis on applique
la stratégie définie dans le pipeline officiel :
- Remplacement par 'Non identifié' pour les colonnes nom et type des acteurs
- Conservation des NaN pour les codes pays (manquants informatifs)

In [4]:
# État des valeurs manquantes avant traitement
print("=== Valeurs manquantes avant traitement ===")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
manquants = pd.DataFrame({'Manquantes': missing, '%': missing_pct})
print(manquants[manquants['Manquantes'] > 0].to_string())

# Remplacement par 'Non identifié'
cols_non_identifie = [
    'Actor1Name', 'Actor2Name',
    'Actor1Type1Code', 'Actor2Type1Code'
]
for col in cols_non_identifie:
    df[col] = df[col].fillna('Non identifié')

# Vérification
print("\n=== Valeurs manquantes après traitement ===")
missing_after = df.isnull().sum()
missing_pct_after = (missing_after / len(df) * 100).round(2)
manquants_after = pd.DataFrame({'Manquantes': missing_after, '%': missing_pct_after})
print(manquants_after[manquants_after['Manquantes'] > 0].to_string())

=== Valeurs manquantes avant traitement ===
                   Manquantes      %
Actor1Name               2246   9.57
Actor1CountryCode       11534  49.16
Actor1Type1Code         12515  53.34
Actor2Name               7199  30.68
Actor2CountryCode       13695  58.37
Actor2Type1Code         15736  67.07

=== Valeurs manquantes après traitement ===
                   Manquantes      %
Actor1CountryCode       11534  49.16
Actor2CountryCode       13695  58.37


### Constat
Avant traitement, 6 colonnes présentaient des valeurs manquantes :

- `Actor1Type1Code` (53.64%) et `Actor2Type1Code` (67.11%) : taux élevés
  car GDELT n'identifie pas toujours le rôle de l'acteur (gouvernement,
  militaire, ONG, etc.). Ces manquants sont normaux — remplacés par
  'Non identifié'.
- `Actor1CountryCode` (48.76%) et `Actor2CountryCode` (57.90%) : conservés
  intentionnellement en NaN. Un manquant ici signifie que l'acteur n'a
  pas pu être rattaché à un pays (acteur générique : "GUNMEN",
  "AUTHORITIES"). Supprimer ces lignes biaiserait l'analyse.
- `Actor1Name` (9.47%) et `Actor2Name` (30.31%) : remplacés par
  'Non identifié' pour uniformiser l'affichage.

Après traitement, seuls les codes pays conservent des NaN —
conformément à la stratégie définie dans le pipeline officiel.

## 3. Enrichissement

On crée les colonnes dérivées nécessaires à l'analyse,
au dashboard et au modèle ML. Aucune ligne n'est supprimée
à cette étape — on ajoute uniquement de l'information.

### 3.1 Colonnes temporelles

Extraction du mois depuis SQLDATE pour permettre
les agrégations temporelles dans le dashboard et les analyses.

In [5]:
# Colonne mois (période mensuelle)
df['mois'] = df['SQLDATE'].dt.to_period('M').astype(str)

print("Distribution mensuelle des événements :")
print(df['mois'].value_counts().sort_index().to_string())

Distribution mensuelle des événements :
mois
2025-01    2127
2025-02    1657
2025-03    2046
2025-04    2089
2025-05    1776
2025-06     888
2025-07    2245
2025-08    1495
2025-09    1594
2025-10    1650
2025-11    1750
2025-12    4144


### Constat
Les 23 461 événements se répartissent sur les 12 mois de 2025.
**Décembre 2025 se détache nettement avec 4 221 événements**
— soit plus du double de la moyenne mensuelle (~1 955).
Ce pic correspond à la tentative de coup d'état du 7 décembre 2025,
qui a suscité un intérêt médiatique international exceptionnel.
**Juin 2025 est le mois le plus calme** (908 événements).

### 3.2 Zone géographique (nord / sud / non localisé)

On crée une colonne `zone_geo` en 3 catégories basées sur la latitude.
Notre EDA a montré que les dynamiques nord/sud sont radicalement
différentes : le nord (Atakora, Alibori) concentre les événements
les plus violents liés à la menace jihadiste aux frontières,
tandis que le sud reste stable et diplomatiquement actif.

Bornes utilisées (coordonnées précises du Bénin) :
- Nord : ActionGeo_Lat >= 10.0°N
- Sud  : ActionGeo_Lat < 10.0°N (hors centroïde générique)
- Non localisé : centroïde générique (9.5°N, 2.25°E) ou NaN

In [6]:
def zone_geo(row):
    lat = row['ActionGeo_Lat']
    lon = row['ActionGeo_Long']
    # Centroïde générique ou NaN
    if pd.isna(lat) or pd.isna(lon):
        return 'Non localisé'
    if round(lat, 1) == 9.5 and round(lon, 2) == 2.25:
        return 'Non localisé'
    # Nord / Sud
    if lat >= 10.0:
        return 'Nord (Atakora / Alibori)'
    else:
        return 'Sud'

df['zone_geo'] = df.apply(zone_geo, axis=1)

print("Distribution zone géographique :")
print(df['zone_geo'].value_counts().to_string())
print(f"\n% Non localisé : {(df['zone_geo'] == 'Non localisé').mean()*100:.1f}%")

Distribution zone géographique :
zone_geo
Non localisé                21406
Sud                          1416
Nord (Atakora / Alibori)      639

% Non localisé : 91.2%


### Constat
Comme anticipé dans le pipeline, **91.2% des événements sont
non localisés** — ils portent le centroïde générique du pays.
C'est une limite inhérente à GDELT pour les pays peu couverts.

Les 8.8% d'événements géolocalisés précisément se répartissent :
- **Sud : 1 449 événements** — zone diplomatique et économique active
- **Nord : 652 événements** — zone frontalière sous pression sécuritaire

Malgré leur faible volume, ces événements précisément localisés
sont les plus informatifs pour l'analyse géographique —
notamment la concentration de violence dans le nord confirmée par l'EDA.

### 3.3 Classification QuadClass et indicateur de violence

On crée deux colonnes lisibles à partir de QuadClass et EventRootCode :
- `type_quadclass` : libellé lisible des 4 grandes familles d'événements
- `is_violent` : booléen indiquant si l'événement est conflictuel (codes 13-20)

Ces colonnes simplifient les filtres du dashboard et rendent
les visualisations compréhensibles sans connaissance de CAMEO.

In [7]:
# Libellé QuadClass
quadclass_labels = {
    1: 'Coopération verbale',
    2: 'Coopération matérielle',
    3: 'Conflit verbal',
    4: 'Conflit matériel'
}
df['type_quadclass'] = df['QuadClass'].map(quadclass_labels).fillna('Non classifié')

# Indicateur de violence
df['is_violent'] = df['EventRootCode'] >= 13

print("Distribution QuadClass :")
print(df['type_quadclass'].value_counts().to_string())
print(f"\n% événements violents : {df['is_violent'].mean()*100:.1f}%")
print(f"Nombre événements violents : {df['is_violent'].sum()}")

Distribution QuadClass :
type_quadclass
Coopération verbale       14946
Conflit matériel           3305
Conflit verbal             2852
Coopération matérielle     2358

% événements violents : 17.1%
Nombre événements violents : 4023


### Constat corrigé
- **Coopération verbale (64.8%)** : dominante.
- **Conflit matériel (14.4%) + Conflit verbal (12.3%)** = **26.7%**
  d'événements conflictuels au sens large — incluant désaccords
  diplomatiques, rejets, désapprobations.
- **17.2% d'événements violents** (EventRootCode >= 13) : sous-ensemble
  plus strict — menaces, coercition, agressions, usage d'armes.
  Ces 17.2% sont inclus dans les 26.7% conflictuels.

### 3.4 Score d'impact pondéré

Le GoldsteinScale est assigné par type d'événement — deux attaques
de tailles très différentes reçoivent le même score. NumArticles
capture le poids médiatique réel de chaque événement.

On combine les deux via un score pondéré :
`impact_pondere = GoldsteinScale × log(NumArticles + 1)`

Le logarithme évite qu'un événement très couvert (48 articles)
n'écrase complètement un événement moins couvert (5 articles).

In [8]:
df['impact_pondere'] = df['GoldsteinScale'] * np.log(df['NumArticles'] + 1)

print("=== Statistiques impact pondéré ===")
print(df['impact_pondere'].describe().round(2))
print(f"\nTop 5 événements à impact positif pondéré :")
print(df.nlargest(5, 'impact_pondere')[
    ['SQLDATE', 'type_quadclass', 'Actor1Name', 'impact_pondere', 'NumArticles']
].to_string(index=False))
print(f"\nTop 5 événements à impact négatif pondéré :")
print(df.nsmallest(5, 'impact_pondere')[
    ['SQLDATE', 'type_quadclass', 'Actor1Name', 'impact_pondere', 'NumArticles']
].to_string(index=False))

=== Statistiques impact pondéré ===
count    23461.00
mean         0.96
std          8.57
min        -34.34
25%         -3.22
50%          2.36
75%          6.44
max         26.00
Name: impact_pondere, dtype: float64

Top 5 événements à impact positif pondéré :
   SQLDATE         type_quadclass     Actor1Name  impact_pondere  NumArticles
2025-02-01 Coopération matérielle      COMPANIES       25.995004           40
2025-02-22 Coopération matérielle THE NETHERLAND       24.475553           32
2025-10-24    Coopération verbale          BENIN       24.356180           20
2025-10-24    Coopération verbale  Non identifié       24.356180           20
2025-02-19    Coopération verbale           BANK       24.356180           20

Top 5 événements à impact négatif pondéré :
   SQLDATE   type_quadclass Actor1Name  impact_pondere  NumArticles
2025-02-21 Conflit matériel      BENIN      -34.339872           30
2025-01-09 Conflit matériel      BENIN      -30.445224           20
2025-01-09 Conflit ma

### Constat
L'impact pondéré moyen est de **+0.95** — légèrement positif,
cohérent avec le Goldstein moyen de l'EDA.

**Top événements positifs :**
- **01 fév. (COMPANIES, +26.0)** : fort accord économique impliquant
  des entreprises — couverture importante (40 articles).
- **22 fév. (Pays-Bas, +24.5)** : restitution des bronzes du Bénin
  royal — événement culturel majeur à fort retentissement (32 articles).

**Top événements négatifs :**
- **21 fév. (BENIN, -34.3)** : l'événement le plus impactant
  négativement de l'année — Conflit matériel avec 30 articles.
  Goldstein = -10 (score minimal) × log(31) ≈ -34.3.
- **Présence d'AL QAEDA (19 avr., -30.4)** : signal fort
  de la menace terroriste dans la région.

L'écart entre min (-34.3) et max (+26.0) illustre bien
la pertinence du score : il différencie les événements
par leur combinaison de gravité ET de visibilité médiatique.

### 3.5 Domaine source

On extrait le nom de domaine depuis SOURCEURL pour identifier
quels médias couvrent le Bénin. Cela permet d'analyser
la provenance géographique et éditoriale de la couverture
médiatique internationale.

In [9]:
df['source_domain'] = df['SOURCEURL'].str.extract(r'https?://(?:www\.)?([^/]+)')

print("=== Top 15 médias couvrant le Bénin (2025) ===")
top_media = df['source_domain'].value_counts().head(15)
total = len(df)
for domain, count in top_media.items():
    print(f"  {domain:<40} {count:>5} ({count/total*100:.1f}%)")

print(f"\nNombre de médias distincts : {df['source_domain'].nunique()}")
print(f"Valeurs manquantes : {df['source_domain'].isna().sum()}")

=== Top 15 médias couvrant le Bénin (2025) ===
  dailypost.ng                              1561 (6.7%)
  punchng.com                               1463 (6.2%)
  nigerianobservernews.com                   982 (4.2%)
  lanouvelletribune.info                     812 (3.5%)
  leadership.ng                              739 (3.1%)
  guardian.ng                                671 (2.9%)
  allafrica.com                              640 (2.7%)
  thesun.ng                                  516 (2.2%)
  saharareporters.com                        511 (2.2%)
  blueprint.ng                               488 (2.1%)
  thisdaylive.com                            478 (2.0%)
  quicknews-africa.net                       428 (1.8%)
  premiumtimesng.com                         402 (1.7%)
  thenationonlineng.net                      377 (1.6%)
  theeagleonline.com.ng                      306 (1.3%)

Nombre de médias distincts : 1323
Valeurs manquantes : 0


### Constat
**1 323 médias distincts** couvrent le Bénin en 2025 — diversité
importante de sources internationales.

Les médias nigérians (.ng) dominent le top 15 avec ~37% de la
couverture totale. Cela s'explique naturellement par la proximité
géographique et les liens étroits entre le Nigeria et le Bénin —
le Nigeria est le pays qui interagit le plus avec le Bénin selon
nos données (Actor1CountryCode NGA = 2ème rang). La presse
nigériane couvre légitimement les événements frontaliers,
diplomatiques et sécuritaires impliquant le Bénin.

`lanouvelletribune.info` (3.5%) est le seul média béninois
francophone dans ce top 15 — signal que la presse locale béninoise
est peu indexée par GDELT, qui favorise les sources anglophones
à large audience.

**Biais identifié et non corrigeable en Phase 1 :** une partie
des articles `.ng` peut concerner Benin City (Nigeria) et non
le Bénin-pays. La distinction nécessiterait une analyse du contenu
des articles (API GDELT DOC) — reportée en Phase 2.

### 3.6 Communauté linguistique de couverture

On détecte la communauté linguistique depuis le domaine de SOURCEURL.
Cette colonne permet d'analyser quelle communauté internationale
couvre le Bénin et avec quel prisme.

Approche : correspondance par domaine URL — approximative mais
suffisante pour identifier les grandes tendances (Phase 1).
La détection exacte via la table Mentions GDELT sera implémentée
en Phase 2.

In [10]:
def detect_language_community(url):
    if pd.isna(url):
        return 'Non identifié'
    url_lower = url.lower()

    if any(x in url_lower for x in [
        '.fr/', 'rfi.fr', 'lemonde', 'jeuneafrique', 'afrique.',
        'lanouvelletribune', 'banouto', 'acotonou', 'beninwebtv',
        'matinlibre', 'fraternite', 'liberation.fr', 'lefigaro',
        'allafrica.com/fr', 'francophone', 'la-croix',
        'rewmi.com', 'lavocedeltrentino', 'ledauphine'
    ]):
        return 'Francophonie'

    elif any(x in url_lower for x in [
        '.ng/', '.gh/', '.ke/', '.za/', '.ug/', '.rw/',
        'bbc.', 'reuters.', 'allafrica.com', 'punchng',
        'dailypost', 'vanguard', 'premiumtimes', 'guardian.ng',
        'thisdaylive', 'leadership.ng', 'thesun.ng',
        'saharareporters', 'blueprint.ng', 'africanews',
        'blackagendareport', 'channelstv', 'theeagleonline',
        'thenationonlineng', 'quicknews-africa', 'brandonsun',
        'kjzz.org', 'legit.ng', 'nigerianobserver'
    ]):
        return 'Anglophone / Commonwealth'

    elif any(x in url_lower for x in [
        '.cn/', 'xinhua', 'chinadaily', 'cgtn',
        'english.news.cn', 'globaltimes'
    ]):
        return 'Chine'

    elif any(x in url_lower for x in [
        '.pt/', '.ao/', '.br/', 'brasil', 'angola',
        'cameroun24', 'afdb.africa'
    ]):
        return 'Lusophonie / Autre Afrique'

    elif any(x in url_lower for x in [
        '.ma/', '.tn/', '.dz/', '.eg/', 'arabic', 'alarab'
    ]):
        return 'Arabophone'

    else:
        return 'Autre / Non identifié'

df['communaute_linguistique'] = df['SOURCEURL'].apply(detect_language_community)

print("=== Distribution communauté linguistique ===")
dist = df['communaute_linguistique'].value_counts()
pct = (dist / len(df) * 100).round(2)
print(pd.DataFrame({'Occurrences': dist, '%': pct}).to_string())
print(f"\n% identifiés : {(df['communaute_linguistique'] != 'Autre / Non identifié').mean()*100:.1f}%")

=== Distribution communauté linguistique ===
                            Occurrences      %
communaute_linguistique                       
Anglophone / Commonwealth         11603  49.46
Autre / Non identifié             10222  43.57
Francophonie                       1230   5.24
Chine                               210   0.90
Lusophonie / Autre Afrique          156   0.66
Arabophone                           40   0.17

% identifiés : 56.4%


### Constat
**56.4% des articles sont identifiés** par communauté linguistique —
43.6% restent non identifiés, ce qui est attendu pour une approche
par correspondance de domaines (1 323 médias distincts, impossible
de tous les couvrir manuellement).

**Anglophone / Commonwealth domine largement (49.5%)** — cohérent
avec la domination des médias nigérians (.ng) observée dans
`source_domain`. Le Nigeria, pays frontalier anglophone, est
le principal vecteur de couverture internationale du Bénin.

**Francophonie (5.2%)** — sous-représentée malgré le fait que
le Bénin soit un pays francophone. Deux explications :
- GDELT indexe davantage les sources anglophones à large audience
- La presse béninoise locale est peu présente dans GDELT

**Chine (0.9%)** — présence modeste mais notable, cohérente avec
les investissements chinois en Afrique de l'Ouest et la couverture
de Xinhua/CGTN sur le continent.

**Arabophone (0.17%)** et **Lusophonie (0.66%)** — présences
marginales mais réelles.

**Limite à documenter :** 43.6% de "Autre / Non identifié"
signifie que cette colonne donne des tendances, pas des certitudes.
La détection exacte via la table Mentions GDELT (champ
`MentionDocTranslationInfo`) reste l'approche cible pour la Phase 2.

## 4. Production des fichiers finaux

On produit les deux fichiers du pipeline officiel dans
`data/processed/` :
- `benin_2025_clean.csv` : dataset complet nettoyé et enrichi
- `benin_2025_agregat_mensuel.csv` : agrégat mensuel pré-calculé
  pour le dashboard

On vérifie d'abord l'état final du dataset avant export.

In [11]:
# ── État final du dataset ────────────────────────────────────
print("=== État final du dataset ===")
print(f"Lignes    : {len(df)}")
print(f"Colonnes  : {len(df.columns)}")
print(f"\nColonnes disponibles :")
for col in df.columns:
    print(f"  - {col}")

print(f"\nValeurs manquantes restantes :")
missing = df.isnull().sum()
print(missing[missing > 0].to_string())

=== État final du dataset ===
Lignes    : 23461
Colonnes  : 26

Colonnes disponibles :
  - SQLDATE
  - EventCode
  - EventBaseCode
  - EventRootCode
  - QuadClass
  - Actor1Name
  - Actor1CountryCode
  - Actor1Type1Code
  - Actor2Name
  - Actor2CountryCode
  - Actor2Type1Code
  - ActionGeo_FullName
  - ActionGeo_Lat
  - ActionGeo_Long
  - GoldsteinScale
  - NumMentions
  - NumArticles
  - AvgTone
  - SOURCEURL
  - mois
  - zone_geo
  - type_quadclass
  - is_violent
  - impact_pondere
  - source_domain
  - communaute_linguistique

Valeurs manquantes restantes :
Actor1CountryCode    11534
Actor2CountryCode    13695


In [12]:
# Créer le dossier processed s'il n'existe pas
os.makedirs('../data/processed', exist_ok=True)

# Export dataset complet
df.to_csv('../data/processed/benin_2025_clean.csv', index=False)

# Vérification
df_check = pd.read_csv('../data/processed/benin_2025_clean.csv')
print(f"✓ benin_2025_clean.csv exporté avec succès")
print(f"  Lignes   : {df_check.shape[0]}")
print(f"  Colonnes : {df_check.shape[1]}")

✓ benin_2025_clean.csv exporté avec succès
  Lignes   : 23461
  Colonnes : 26


### Constat
`benin_2025_clean.csv` exporté avec succès —
**23 461 lignes et 25 colonnes** confirmées en relecture.

### 4.2 benin_2025_agregat_mensuel.csv

On calcule l'agrégat mensuel pré-calculé pour le dashboard.
Ce fichier regroupe les indicateurs clés par mois et par zone
géographique pour éviter les recalculs à chaque chargement
du dashboard Streamlit.

In [13]:
agregat = df.groupby(['mois', 'zone_geo']).agg(
    nb_evenements=('SQLDATE', 'count'),
    goldstein_moyen=('GoldsteinScale', 'mean'),
    ton_moyen=('AvgTone', 'mean'),
    impact_moyen=('impact_pondere', 'mean'),
    pct_violent=('is_violent', 'mean'),
    nb_articles_total=('NumArticles', 'sum'),
    nb_sources=('source_domain', 'nunique')
).reset_index()

agregat['pct_violent'] = (agregat['pct_violent'] * 100).round(2)
agregat = agregat.round(3)
agregat = agregat.sort_values(['mois', 'zone_geo'])

# Export
agregat.to_csv('../data/processed/benin_2025_agregat_mensuel.csv', index=False)

# Vérification
agregat_check = pd.read_csv('../data/processed/benin_2025_agregat_mensuel.csv')
print(f"✓ benin_2025_agregat_mensuel.csv exporté avec succès")
print(f"  Lignes   : {agregat_check.shape[0]}")
print(f"  Colonnes : {agregat_check.shape[1]}")
print(f"\nAperçu :")
display(agregat_check.head(10))

✓ benin_2025_agregat_mensuel.csv exporté avec succès
  Lignes   : 36
  Colonnes : 9

Aperçu :


,mois,zone_geo,nb_evenements,goldstein_moyen,ton_moyen,impact_moyen,pct_violent,nb_articles_total,nb_sources
0,2025-01,Non localisé,1857,0.310,-1.633,0.589,14.75,11443,265
1,2025-01,Nord (Atakora / Alibori),118,-3.153,-6.638,-4.982,47.46,593,34
2,2025-01,Sud,152,2.304,2.004,4.130,2.63,976,40
3,2025-02,Non localisé,1477,0.332,-1.723,0.767,18.69,9344,208
4,2025-02,Nord (Atakora / Alibori),92,-1.035,-4.721,-1.233,35.87,503,18
5,2025-02,Sud,88,0.565,-2.352,1.433,12.50,571,21
6,2025-03,Non localisé,1840,0.464,-1.957,0.860,17.50,10511,182
7,2025-03,Nord (Atakora / Alibori),64,-3.088,-5.615,-5.234,48.44,342,15
8,2025-03,Sud,142,2.010,0.405,2.839,6.34,715,22
9,2025-04,Non localisé,1910,0.767,-0.877,1.028,16.28,11261,208


In [14]:
print("=== Agrégat complet par zone ===")
print(agregat.to_string(index=False))

print("\n=== Moyennes annuelles par zone ===")
print(agregat.groupby('zone_geo').agg(
    goldstein_moyen=('goldstein_moyen', 'mean'),
    ton_moyen=('ton_moyen', 'mean'),
    pct_violent=('pct_violent', 'mean'),
    nb_evenements=('nb_evenements', 'sum')
).round(2).to_string())

=== Agrégat complet par zone ===
   mois                 zone_geo  nb_evenements  goldstein_moyen  ton_moyen  impact_moyen  pct_violent  nb_articles_total  nb_sources
2025-01             Non localisé           1857            0.310     -1.633         0.589        14.75              11443         265
2025-01 Nord (Atakora / Alibori)            118           -3.153     -6.638        -4.982        47.46                593          34
2025-01                      Sud            152            2.304      2.004         4.130         2.63                976          40
2025-02             Non localisé           1477            0.332     -1.723         0.767        18.69               9344         208
2025-02 Nord (Atakora / Alibori)             92           -1.035     -4.721        -1.233        35.87                503          18
2025-02                      Sud             88            0.565     -2.352         1.433        12.50                571          21
2025-03             Non local

### Constat
`benin_2025_agregat_mensuel.csv` exporté avec succès —
**36 lignes** (12 mois × 3 zones) et **9 colonnes**.

Les moyennes annuelles par zone confirment et quantifient
la fracture nord/sud identifiée dans l'EDA :

**Nord (Atakora / Alibori) :**
- Goldstein moyen annuel : **-1.40** — seule zone en territoire
  négatif, instabilité structurelle sur toute l'année
- % violent moyen : **35.47%** — plus d'1 événement sur 3
  est violent, contre 16% pour le reste du pays
- Pic critique en avril : Goldstein **-5.49**, pct_violent **61.6%**
  et impact_moyen **-10.08** — mois le plus conflictuel de l'année
- Seulement **652 événements** au total — zone peu couverte
  mais événements graves

**Sud :**
- Goldstein moyen annuel : **+1.54** — zone stable et positive
- % violent moyen : **10.46%** — 3x moins violent que le nord
- Ton moyen : **+0.07** — seule zone avec couverture neutre
  à légèrement positive
- **1 416 événements** — zone diplomatiquement et économiquement active

**Non localisé (91% des événements) :**
- Goldstein moyen : **+0.63** — stabilité globale apparente
- % violent : **16.43%** — moyenne nationale
- Décembre 2025 domine avec **3 855 événements** non localisés
  — pic lié à la tentative de coup d'état du 7 décembre

**Insight clé pour le dashboard :**
Le Goldstein national moyen (+0.63) masque une réalité
géographique radicalement différente entre un nord instable
(-1.40) et un sud stable (+1.54). Un décideur public
qui ne regarde que l'indicateur national serait trompé
sur la situation réelle du pays.

## 5. Dictionnaire des colonnes

### benin_2025_clean.csv — 26 colonnes

#### Colonnes originales GDELT

| Colonne | Type | Description |
|---------|------|-------------|
| `SQLDATE` | datetime | Date de l'événement (converti depuis YYYYMMDD) |
| `EventCode` | int | Code CAMEO complet de l'événement (niveau le plus granulaire, 200+ codes) |
| `EventBaseCode` | int | Code CAMEO niveau intermédiaire — agrège EventCode en sous-catégories |
| `EventRootCode` | int | Code CAMEO niveau racine (1-20) — 20 grandes familles d'événements |
| `QuadClass` | int | Classification en 4 familles : 1=Coopération verbale, 2=Coopération matérielle, 3=Conflit verbal, 4=Conflit matériel |
| `Actor1Name` | str | Nom du premier acteur (pays, organisation, groupe, individu). NaN → 'Non identifié' |
| `Actor1CountryCode` | str | Code pays CAMEO (3 lettres) de l'acteur 1. NaN conservé (manquant informatif) |
| `Actor1Type1Code` | str | Rôle de l'acteur 1 : GOV=gouvernement, MIL=militaire, OPP=opposition, REB=rebelles. NaN → 'Non identifié' |
| `Actor2Name` | str | Nom du second acteur. NaN → 'Non identifié' |
| `Actor2CountryCode` | str | Code pays CAMEO de l'acteur 2. NaN conservé (manquant informatif) |
| `Actor2Type1Code` | str | Rôle de l'acteur 2. NaN → 'Non identifié' |
| `ActionGeo_FullName` | str | Nom lisible du lieu de l'action (ex : 'Kandi, Alibori, Benin') |
| `ActionGeo_Lat` | float | Latitude GPS du lieu. 9.5°N = centroïde générique (lieu non identifié précisément) |
| `ActionGeo_Long` | float | Longitude GPS du lieu. 2.25°E = centroïde générique |
| `GoldsteinScale` | float | Score de stabilité (-10 à +10). Assigné par type d'événement, pas par événement individuel |
| `NumMentions` | int | Nombre total de mentions de l'événement dans la fenêtre de 15 minutes GDELT |
| `NumArticles` | int | Nombre d'articles sources distincts couvrant l'événement — proxy du poids médiatique |
| `AvgTone` | float | Ton médiatique moyen (-100 à +100). Valeurs habituelles entre -10 et +10 |
| `SOURCEURL` | str | URL du premier article source ayant rapporté l'événement |

#### Colonnes dérivées

| Colonne | Type | Description |
|---------|------|-------------|
| `mois` | str | Période mensuelle au format YYYY-MM, extraite de SQLDATE |
| `zone_geo` | str | Zone géographique : 'Nord (Atakora / Alibori)', 'Sud', 'Non localisé' |
| `type_quadclass` | str | Libellé lisible de QuadClass : 'Coopération verbale', 'Coopération matérielle', 'Conflit verbal', 'Conflit matériel' |
| `is_violent` | bool | Vrai si EventRootCode >= 13 (événement conflictuel ou violent selon CAMEO) |
| `impact_pondere` | float | Score composite : GoldsteinScale × log(NumArticles + 1). Combine gravité et visibilité médiatique |
| `source_domain` | str | Nom de domaine extrait de SOURCEURL (ex : allafrica.com, rfi.fr) |
| `communaute_linguistique` | str | Communauté linguistique de la source : 'Anglophone / Commonwealth', 'Francophonie', 'Chine', 'Lusophonie / Autre Afrique', 'Arabophone', 'Autre / Non identifié'. Détection approximative par domaine URL — Phase 2 utilisera la table Mentions GDELT |

---

### benin_2025_agregat_mensuel.csv — 9 colonnes

| Colonne | Type | Description |
|---------|------|-------------|
| `mois` | str | Période mensuelle (YYYY-MM) |
| `zone_geo` | str | Zone géographique (3 catégories) |
| `nb_evenements` | int | Nombre d'événements sur le mois et la zone |
| `goldstein_moyen` | float | Score Goldstein moyen — indicateur de stabilité perçue |
| `ton_moyen` | float | Ton médiatique moyen — indicateur de sentiment de couverture |
| `impact_moyen` | float | Impact pondéré moyen — combine stabilité et visibilité médiatique |
| `pct_violent` | float | Pourcentage d'événements violents (EventRootCode >= 13) |
| `nb_articles_total` | int | Volume médiatique total sur le mois et la zone |
| `nb_sources` | int | Nombre de médias distincts ayant couvert le Bénin sur le mois et la zone |